**Week 4 Learning Unkown Summary Stats** \
Date: 7.29 \
Objectives:
- use embedding network defined from last week to learn summary stats of more complex distributions with unknown summaries.
    - Gaussian Mixture
    - Lotka-Volterra (predator-prey)

**Gaussian Mixture** \
This example uses a linear combination of two Gaussians, where the relative weighting $\pi$, and respective means and standard deviations are sampled from a flat 5D prior. The sampled parameters are then used to simulate data ($x_1 \sim N(\mu_1, \sigma_1^2)$, $x_2 \sim N(\mu_2, \sigma_2^2)$ and $x = x_1 + x_2$, the combined set). \
A 3 layer sequential network (Linear --> ReLU --> Linear) is used to learned the summaries, which is compared to a proposed optimal summary of $\pi*\mu_1 + (1-\pi)*\mu_2$, using cosine simularity and Pearson correlation as metrics to validate this proposal.

In [ ]:
# Gaussian Mixture
import torch
from sbi import utils as utils
from sbi.inference import simulate_for_sbi
from sbi.utils.user_input_checks import prepare_for_sbi
from sbi.inference import SNPE
import matplotlib.pyplot as plt
import numpy as np
from sbi.neural_nets import posterior_nn
import torch.nn as nn
import torch.nn.functional as F
torch.random.manual_seed(13)

dim = 5 # parameter dim
n_obs = 1000
prior = utils.BoxUniform(low=torch.zeros(dim), high=torch.ones(dim)*3)
# x ~ pi*N(mu1, sd1 ** 2) + (1-pi)*N(mu2,sd**2)
# abitary parameter choices
# theta = [pi, mu1, mu2, sd1, sd2]

# simulator 
def simulator(theta):
    pi = theta[:, 0]
    mu1 = theta[:, 1]
    sd1 = theta[:, 2]
    mu2 = theta[:, 3]
    sd2 = theta[:, 4]
    
    n_batch = theta.shape[0]
    
    x1 = torch.normal(mu1.view(-1, 1), sd1.view(-1, 1).clamp(min=1e-2)).to(theta)
    x2 = torch.normal(mu2.view(-1, 1), sd2.view(-1, 1).clamp(min=1e-2)).to(theta)
    
    pi = pi.view(-1, 1)
    x = pi * x1 + (1 - pi) * x2
    x = x.repeat(1, n_obs)

    return x, x1, x2

# 3 layer embedding network
class embedding_network(nn.Module):
    def __init__(self, input_dim, embedding_dim):
        super().__init__()
        self.network = nn.Sequential(  
            nn.Linear(input_dim, 64),   
            nn.ReLU(),
            nn.Linear(64, embedding_dim)   
        )

    def forward(self, x):
        return self.network(x)

# training sbi, regular training method
def train_sbi(net, num_simulations, prior):
    sim, pri = prepare_for_sbi(lambda theta: simulator(theta)[0], prior)

    inference = SNPE(prior=pri, density_estimator=posterior_nn(model='maf', embedding_net=net))

    theta_single = pri.sample((1,))
    theta = theta_single.repeat(num_simulations, 1)
    print(f'parameters [pi, mu1, mu2, sd1, sd2]:{theta_single}')

    x, x1, x2 = simulator(theta)
    x = x.view(x.shape[0], -1)  # flatten for input to network

    density_estimator = inference.append_simulations(theta, x).train()
    posterior = inference.build_posterior(density_estimator)

    return posterior, net, theta, x, x1, x2

Since the optimal summaries of this mixture distribution is unknown and the two distributions present are independent, it's sensible to start with some linear combination of the individual means as a first guess of the unknown summary.

In [13]:
# try the same linear combo, overall mean = pi * mean1 + (1-pi) * mean2

class analyze_embeddings:
    def __init__(self, embedding_net, x_eval, optimal_summary, x1, x2):
        self.embedding_net = embedding_net
        self.eval_x = x_eval
        self.optimal_summary = optimal_summary
        self.x1 = x1
        self.x2 = x2
        self.analyze()

    def analyze(self):
        learned_summary = self.embedding_net(self.eval_x)
        optimal_summary = self.optimal_summary(self.eval_x, self.x1, self.x2)

        self.learned_summary = learned_summary
        self.optimal_summary = optimal_summary

    # project learned summary to 1D, compare the first column component only
        projected_learned = learned_summary[:, 0:1]
        self.projected_learned = projected_learned

    # cosine similarity
        learned_norm = F.normalize(projected_learned, dim=1)
        optimal_norm = F.normalize(optimal_summary, dim=1)
        cosine_sim = torch.sum(learned_norm * optimal_norm, dim=1).mean().item()

    # pearson correlation
        corr = torch.corrcoef(torch.stack([
        projected_learned.squeeze(),
        optimal_summary.squeeze()
        ]))[0, 1].item()

        print(f"Cosine similarity: {cosine_sim:.4f}")
        print(f"Pearson correlation: {corr:.4f}")

        self.cosine_similarity = cosine_sim
        self.correlation = corr


def unknown_summary_experiment():

    embedding_dims = [1, 2, 4, 8]
    results = {}

    for dim_embed in embedding_dims:
        print(f"\nTraining with embedding_dim = {dim_embed}")
        embed_net = embedding_network(input_dim=n_obs, embedding_dim=dim_embed)
        posterior, net, theta, x, x1, x2 = train_sbi(embed_net, num_simulations=n_obs, prior=prior)

        def optimal_summary_trial(theta, x1, x2):
            # Compute optimal summary from original x1, x2, theta
            pi = theta[:100, 0]
            mean1 = x1[:100].mean(dim=1)
            mean2 = x2[:100].mean(dim=1)
            return (pi * mean1 + (1 - pi) * mean2).unsqueeze(1)
        
        analysis = analyze_embeddings(net, x[:100], optimal_summary_trial, x1, x2)
        results[dim_embed] = {
            "cosine_similarity": analysis.cosine_similarity,
            "correlation": analysis.correlation
        }

unknown_summary_experiment()


Training with embedding_dim = 1
parameters [pi, mu1, mu2, sd1, sd2]:tensor([[0.4552, 0.0672, 1.3036, 0.2877, 0.1741]])


C:\Users\boat\AppData\Local\Temp\ipykernel_11740\266995400.py:55: DeprecationWarning: This method is deprecated as of sbi version v0.23.0. and will be removed in a         future release.Please use `process_prior` and `process_simulator` in the future.
  sim, pri = prepare_for_sbi(lambda theta: simulator(theta)[0], prior)


 Neural network successfully converged after 123 epochs.Cosine similarity: 0.9800
Pearson correlation: 0.8961

Training with embedding_dim = 2
parameters [pi, mu1, mu2, sd1, sd2]:tensor([[0.0785, 1.9863, 1.4579, 0.1480, 1.2093]])
 Neural network successfully converged after 923 epochs.Cosine similarity: -0.1000
Pearson correlation: 0.4009

Training with embedding_dim = 4
parameters [pi, mu1, mu2, sd1, sd2]:tensor([[2.6484, 2.9118, 1.6986, 0.5303, 1.9893]])
 Neural network successfully converged after 1360 epochs.Cosine similarity: -1.0000
Pearson correlation: -0.9183

Training with embedding_dim = 8
parameters [pi, mu1, mu2, sd1, sd2]:tensor([[1.5546, 1.9822, 1.0741, 0.7964, 1.2135]])
 Neural network successfully converged after 351 epochs.Cosine similarity: 0.9800
Pearson correlation: 0.8923


cosine simularities are close/equal to 1.0, except for 2D, which means a good alignment between the learned and proposed summaries, the fluctuations in Pearson correlation coefficients could have come from the non-linearity of the network (ReLU layers) but overall the two summaries seem to have a somewhat good linear correspondence.

In [5]:
# run 2D case again and use the second column for analysis
class analyze_embeddings:
    def __init__(self, embedding_net, x_eval, optimal_summary, x1, x2):
        self.embedding_net = embedding_net
        self.eval_x = x_eval
        self.optimal_summary = optimal_summary
        self.x1 = x1
        self.x2 = x2
        self.analyze()

    def analyze(self):
        learned_summary = self.embedding_net(self.eval_x)
        optimal_summary = self.optimal_summary(self.eval_x, self.x1, self.x2)

        self.learned_summary = learned_summary
        self.optimal_summary = optimal_summary

    # project learned summary to 1D, compare the first column component only
        projected_learned = learned_summary[:, 1:2]
        self.projected_learned = projected_learned

    # cosine similarity
        learned_norm = F.normalize(projected_learned, dim=1)
        optimal_norm = F.normalize(optimal_summary, dim=1)
        cosine_sim = torch.sum(learned_norm * optimal_norm, dim=1).mean().item()

    # pearson correlation
        corr = torch.corrcoef(torch.stack([
        projected_learned.squeeze(),
        optimal_summary.squeeze()
        ]))[0, 1].item()

        print(f"Cosine similarity: {cosine_sim:.4f}")
        print(f"Pearson correlation: {corr:.4f}")

        self.cosine_similarity = cosine_sim
        self.correlation = corr


def unknown_summary_experiment():

    embedding_dims = [2]
    results = {}

    for dim_embed in embedding_dims:
        print(f"\nTraining with embedding_dim = {dim_embed}")
        embed_net = embedding_network(input_dim=n_obs, embedding_dim=dim_embed)
        posterior, net, theta, x, x1, x2 = train_sbi(embed_net, num_simulations=n_obs, prior=prior)

        def optimal_summary_trial(theta=torch.tensor([0.0785, 1.9863, 1.4579, 0.1480, 1.2093]), x1=x1, x2=x2):
            # Compute optimal summary from original x1, x2, theta
            pi = theta[:100, 0]
            mean1 = x1[:100].mean(dim=1)
            mean2 = x2[:100].mean(dim=1)
            return (pi * mean1 + (1 - pi) * mean2).unsqueeze(1)
        
        analysis = analyze_embeddings(net, x[:100], optimal_summary_trial, x1, x2)
        results[dim_embed] = {
            "cosine_similarity": analysis.cosine_similarity,
            "correlation": analysis.correlation
        }

unknown_summary_experiment()


Training with embedding_dim = 2
parameters [pi, mu1, mu2, sd1, sd2]:tensor([[2.1072, 2.1695, 1.8058, 0.4027, 1.0215]])


C:\Users\boat\AppData\Local\Temp\ipykernel_12448\266995400.py:55: DeprecationWarning: This method is deprecated as of sbi version v0.23.0. and will be removed in a         future release.Please use `process_prior` and `process_simulator` in the future.
  sim, pri = prepare_for_sbi(lambda theta: simulator(theta)[0], prior)


 Neural network successfully converged after 243 epochs.Cosine similarity: 0.9400
Pearson correlation: 0.9323


Looking at the second component of the learned summaries (with the same parameter input) gave metric values much closer to 1, which indicated the second component encodes more info about the sample summaries and that the proposal summary is validated for 2D case also.

**Lotka-Volterra Model (Predator-Prey model)** \
The model consists of two coupled DEs, $\frac{dx}{dt} = \alpha x - \beta xy$ and $\frac{dy}{dt} = \delta xy - \gamma y$. Here the learned parameters will be the rates ($\alpha, \gamma$) and interaction strength coefficients $\beta, \delta$.

In [6]:
#LV model
import torch
from torchdiffeq import odeint
from sbi import utils as utils
from sbi.inference import simulate_for_sbi
from sbi.utils.user_input_checks import prepare_for_sbi
from sbi.inference import SNPE
from sbi.neural_nets import posterior_nn
import torch.nn as nn
import torch.nn.functional as F
torch.random.manual_seed(13)

# paramters and prior
num_sim = 10000
dim_param = 4
prior = utils.BoxUniform(low=torch.zeros(dim_param), high=torch.ones(dim_param))

# LV model
def lv(theta, t, y): 
    alpha, beta, delta, gamma = theta # theta is a 4 component tensor
    prey, predator = y # x, y = prey, predator
    dprey_dt = alpha*prey - beta*prey*predator
    dpredator_dt = delta*prey*predator - gamma*predator
    return torch.tensor(dprey_dt, dpredator_dt)

# LV simulator
def lv_simulator(theta):
    y0 = torch.tensor([1., 1.])  #initial values
    t = torch.linspace(0, 10, 1000)  #1000 time points between 1 and 10s
    sol = odeint(lambda y, t: lv(theta, t, y), y0, t)
    return sol.flatten()  #flatten to pass through training network

# 3 layer embedding network, same as before for now, might change to CNN/RNN for better time series learning
class embedding_network(nn.Module):
    def __init__(self, input_dim, embedding_dim):
        super().__init__()
        self.network = nn.Sequential(    
            nn.Linear(input_dim, 64),  
            nn.ReLU(),
            nn.Linear(64, embedding_dim)  
        )

    def forward(self, x):
        return self.network(x)
    

# training sbi to learn summaries
def train_sbi(net, num_sim, prior):
    sim, prior = prepare_for_sbi(lv_simulator, prior)

    inference = SNPE(prior=prior, density_estimator=posterior_nn(model='maf', embedding_net=net))

    theta_single = prior.sample((1,))
    print('true parameters:',theta)
    theta = theta_single.repeat(num_sim, 1)

    y = sim(theta_single)
    y = y.view(y.shape[0], -1)  # flatten for input to 
    
    de = inference.append(theta,y).train()
    posterior = inference.build_posterior(de)
    
    return posterior, net, theta, y

# class analyze_embeddings:
#     def __init__(self, embedding_net, x_eval, optimal_summary):
#         self.embedding_net = embedding_net
#         self.eval_x = x_eval
#         self.optimal_summary = optimal_summary
#         self.analyze()

#     def analyze(self):
#         learned_summary = self.embedding_net(self.eval_x)
#         optimal_summary = self.optimal_summary(self.eval_x)

#         self.learned_summary = learned_summary
#         self.optimal_summary = optimal_summary

#     # project learned summary to 1D, compare the first column component only
#         projected_learned = learned_summary[:, 0:1]
#         self.projected_learned = projected_learned

#     # cosine similarity
#         learned_norm = F.normalize(projected_learned, dim=1)
#         optimal_norm = F.normalize(optimal_summary, dim=1)
#         cosine_sim = torch.sum(learned_norm * optimal_norm, dim=1).mean().item()

#     # pearson correlation
#         corr = torch.corrcoef(torch.stack([
#         projected_learned.squeeze(),
#         optimal_summary.squeeze()
#         ]))[0, 1].item()

#         print(f"Cosine similarity: {cosine_sim:.4f}")
#         print(f"Pearson correlation: {corr:.4f}")

#         self.cosine_similarity = cosine_sim
#         self.correlation = corr

# def lv_experiment():
#     def optimal_summary(y):
#         #some parameter tensors
#         return  #some tensor
    
#     embedding_dims = [1, 2, 4, 8]
#     results = {}

#     for dim_embed in embedding_dims:
#         print(f"\nTraining with embedding_dim = {dim_embed}")
#         embed_net = embedding_network(input_dim=num_sim, embedding_dim=dim_embed)
#         posterior, net, theta, x = train_sbi(embed_net, num_simulations=num_sim, prior=prior)

#         analysis = analyze_embeddings(net, y[:100], optimal_summary)
#         results[dim_embed] = {
#             "cosine_similarity": analysis.cosine_similarity,
#             "correlation": analysis.correlation
#         }
    
#     return results


ModuleNotFoundError: No module named 'torchdiffeq'

^^ in all previous cases we had 1 summary/parameter to compare only but this LV case I think all 4 should be taken into accound unless some other single but informative summary is known. Then the network dims should really be >5D and there might be some random correpondence between #coloum to the parameter position?